In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [49]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Read Data

In [12]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
gendered1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')
gendered1 = gendered1[['s1_s2', 'LETTER_GENDER']]
gendered1 = gendered1.rename(columns={'LETTER_GENDER':'label'})

In [13]:
gendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')
gendered2 = gendered2[['s1_s2', 'applicant_gender']]
gendered2 = gendered2.rename(columns={'TEXT':'LETTERTEXT', 'applicant_gender':'label'})

In [14]:
df = pd.concat([gendered1, gendered2], ignore_index=True)

In [15]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [16]:
df['label'] = df['label'].replace(gender_label_mapping)

<ipython-input-16-cc45885305bc>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace(gender_label_mapping)


# Get Unique Words

In [17]:
def tokenize(text):

    tokens = re.findall(r'\b\w+\b', text.lower())
    return set(tokens)

df['tokens'] = df['s1_s2'].apply(tokenize)

In [18]:
male_tokens = set().union(*df[df['label'] == 1]['tokens'])
female_tokens = set().union(*df[df['label'] == 0]['tokens'])

In [19]:
unique_to_male = male_tokens - female_tokens
unique_to_female = female_tokens - male_tokens

In [20]:
unique_to_male

{'hyaluronic',
 'fleet',
 'proteinuria',
 'utensil',
 'sctivities',
 'tugging',
 'sci',
 'beingnamed',
 'guick',
 'grandeur',
 'saxa',
 'sanitary',
 'iteration',
 'instutition',
 'msats',
 'mai',
 'monica',
 'nato',
 'salisbury',
 'fidently',
 'chipping',
 'nonreactive',
 'pathetic',
 'prelimbic',
 'hypotonia',
 'malcolm',
 'hoos',
 'thave',
 'northstate',
 'christiana',
 'bedridden',
 'squared',
 'srbr',
 'ancsthesia',
 'basin',
 'cigna',
 'novelty',
 'preplanned',
 'topping',
 'mrnas',
 'shooter',
 'diversions',
 'sabbatical',
 'ag',
 'waterloo',
 'osteocyte',
 'zucker',
 '68',
 'famed',
 'jight',
 'regrettable',
 'hypobaric',
 'consumers',
 'organiza',
 'mate',
 'ua',
 'contradictions',
 'anesthesiologv',
 'orphanages',
 'guileless',
 'defers',
 'ecutive',
 'rothman',
 'penitentiary',
 'hypomagnesemia',
 'unswerving',
 'spaceflight',
 'ash',
 'persue',
 'drainages',
 'bounced',
 'angle',
 'slipping',
 'interviewees',
 'hum',
 'goodrich',
 'ici',
 'estimations',
 'refresh',
 'respond

In [21]:
unique_to_female

{'healthstart',
 'communcation',
 'judiciary',
 'myasthenic',
 'eisner',
 'killeen',
 'plosone',
 'motorcycle',
 'lgtbq',
 'aplog',
 'mee',
 'anastomose',
 'endavours',
 'myasthenia',
 'requisites',
 'ä',
 'activites',
 'forgiven',
 'iberian',
 'groundedness',
 'bms',
 'cpap',
 '373',
 'misunderstanding',
 'disuse',
 'terpret',
 '408',
 'stares',
 'clip',
 'lps',
 'steak',
 'vaccinathon',
 'handwashing',
 'amvery',
 'antiemetics',
 'northside',
 'umle',
 'dislikes',
 'cex',
 'unworthy',
 'kuskokwim',
 'rector',
 'prohibited',
 'otentially',
 'isacon',
 'ornaments',
 'clinteal',
 'marketable',
 'rsom',
 'stack',
 'iritis',
 'adrenal',
 'heralded',
 'kpc',
 'nephrologist',
 'dispassionate',
 'rack',
 '452',
 'spotted',
 'decortication',
 'diehard',
 'generalsurgery',
 'outpaces',
 'denotes',
 'doffed',
 'sjogren',
 'designating',
 'scrambling',
 'dismissals',
 'interrogated',
 'pdfs',
 'aswell',
 'bombs',
 'wadiisease',
 'kimmel',
 'vying',
 'spiral',
 'sexfw',
 'exemption',
 'obituary',

# TF-IDF

In [24]:
male_texts = df[df['label'] == 1]['s1_s2']
female_texts = df[df['label'] == 0]['s1_s2']

In [25]:
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', max_features=5000)

In [26]:
male_tfidf = vectorizer.fit_transform(male_texts)
male_vocab = vectorizer.get_feature_names_out()
male_scores = male_tfidf.mean(axis=0).A1

female_tfidf = vectorizer.fit_transform(female_texts)
female_vocab = vectorizer.get_feature_names_out()
female_scores = female_tfidf.mean(axis=0).A1


In [27]:
male_df = pd.DataFrame({'token': male_vocab, 'male_score': male_scores})
female_df = pd.DataFrame({'token': female_vocab, 'female_score': female_scores})

tfidf_df = pd.merge(male_df, female_df, on='token', how='outer').fillna(0)

tfidf_df['score_diff'] = tfidf_df['male_score'] - tfidf_df['female_score']

top_male_tokens = tfidf_df.sort_values(by='score_diff', ascending=False).head(20)
top_female_tokens = tfidf_df.sort_values(by='score_diff').head(20)


In [28]:
top_male_tokens

,token,male_score,female_score,score_diff
3304,mr,0.026335,0.000295,0.026039
2045,first_name,0.105659,0.087074,0.018586
3073,man,0.005402,0.000330,0.005072
5598,äö,0.021737,0.019430,0.002307
723,calm,0.007770,0.005907,0.001864
4772,staff,0.017290,0.015465,0.001825
5588,young,0.007450,0.005746,0.001704
4617,showed,0.012498,0.010834,0.001664
3768,physician,0.015756,0.014116,0.001640
2955,liked,0.007661,0.006214,0.001447


In [29]:
top_female_tokens

,token,male_score,female_score,score_diff
3307,ms,0.002079,0.028230,-0.026151
2478,identifier,0.154875,0.160422,-0.005546
2344,health,0.010667,0.014453,-0.003786
5544,woman,0.000348,0.003688,-0.003340
3221,middle_name,0.015395,0.018397,-0.003002
4319,research,0.024801,0.027745,-0.002945
5545,women,0.000821,0.003224,-0.002403
4955,surgery,0.015753,0.017974,-0.002222
3587,outstanding,0.016482,0.018648,-0.002165
753,care,0.030758,0.032505,-0.001747


# TF-IDF on Two Male and Female Documents

In [50]:
male_doc = ' '.join(df[df['label'] == 1]['s1_s2'].tolist())
female_doc = ' '.join(df[df['label'] == 0]['s1_s2'].tolist())

corpus = [male_doc, female_doc]
labels = ['male', 'female']

In [52]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform([male_doc, female_doc])
tokens = vectorizer.get_feature_names_out()

counts = pd.DataFrame(X.toarray().T, index=tokens, columns=['male', 'female'])

counts += 1e-5

counts['log_ratio'] = np.log(counts['male'] / counts['female'])

top_male = counts.sort_values(by='log_ratio', ascending=False).head(40)
top_female = counts.sort_values(by='log_ratio').head(40)

print("Top male-associated tokens:")
print(top_male)

print("\nTop female-associated tokens:")
print(top_female)

Top male-associated tokens:
                   male   female  log_ratio
squadron       38.00001  0.00001  15.150512
guy            35.00001  0.00001  15.068274
scout          32.00001  0.00001  14.978662
eagle          27.00001  0.00001  14.808763
dylan          26.00001  0.00001  14.771022
2d             26.00001  0.00001  14.771022
jin            24.00001  0.00001  14.690980
lcdr           22.00001  0.00001  14.603968
undersea       20.00001  0.00001  14.508658
yong           20.00001  0.00001  14.508658
gas            19.00001  0.00001  14.457365
ordering       16.00001  0.00001  14.285515
reese          16.00001  0.00001  14.285515
aviation       16.00001  0.00001  14.285515
huntington     16.00001  0.00001  14.285515
outdoor        15.00001  0.00001  14.220976
saving         15.00001  0.00001  14.220976
chung          15.00001  0.00001  14.220976
feliciano      15.00001  0.00001  14.220976
roswell        15.00001  0.00001  14.220976
bosh           15.00001  0.00001  14.220976
sean